In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import datasets, layers, models
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

In [2]:
(X_train, y_train), (X_test, y_test)= keras.datasets.mnist.load_data() #loading mnist dataset which includes images of handwritten digits

In [3]:
X_train=X_train/255 #sclading down the data to improve accuracy
X_test = X_test/255

In [25]:
cnn(np.random.randn(1,28,28,1).astype(np.float32))


<tf.Tensor: shape=(1, 10), dtype=float32, numpy=
array([[5.2207030e-05, 1.9329465e-05, 1.0491390e-02, 3.8699996e-01,
        1.0834970e-05, 6.4945879e-04, 8.6197770e-06, 4.7151968e-04,
        6.0129201e-01, 4.7428016e-06]], dtype=float32)>

In [32]:
# Original Sequential model
cnn = tf.keras.Sequential([
    tf.keras.Input(shape=(28,28,1)),
    tf.keras.layers.Conv2D(28, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Conv2D(28, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

# Build by calling once
cnn(np.random.randn(1,28,28,1).astype(np.float32))

# Wrap in Functional API
inputs = tf.keras.Input(shape=(28,28,1), name="input")
x = cnn(inputs)
cnn_func = tf.keras.Model(inputs=inputs, outputs=x)

In [42]:
cnn_func.compile(optimizer = 'adam', loss = 'sparse_categorical_crossentropy', metrics = ['accuracy'])
cnn_func.fit(X_train, y_train, epochs=1)

1875/1875 ━━━━━━━━━━━━━━━━━━━━ 73s 37ms/step - accuracy: 0.9841 - loss: 0.0521


In [49]:
cnn_func.evaluate(X_test,y_test) # checking model accuracy for test dataset

313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 17ms/step - accuracy: 0.9867 - loss: 0.0391


[0.039148423820734024, 0.9866999983787537]

In [43]:
spec = (tf.TensorSpec((None,28,28,1), tf.float32, name="input"),)
onnx_path = "cnn_model.onnx"
model_proto, _ = tf2onnx.convert.from_keras(cnn_func, input_signature=spec, output_path=onnx_path)
opset=16
print(f"ONNX model saved at {onnx_path}")

ONNX model saved at cnn_model.onnx


In [44]:
import onnx
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
print("ONNX model is valid!")

ONNX model is valid!


In [45]:
import onnxruntime as ort

session = ort.InferenceSession("cnn_model.onnx")
input_name = session.get_inputs()[0].name

In [46]:
import numpy as np

predictions = []

for img in X_test:
    
    img = img.reshape(1,28,28,1).astype(np.float32)
    
    pred = session.run(None, {input_name: img})
    
    predicted_class = np.argmax(pred[0])
    
    predictions.append(predicted_class)

In [47]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, predictions)

print("ONNX model accuracy:", accuracy)

ONNX model accuracy: 0.9867


In [50]:
pip install onnxruntime onnxruntime-tools

Note: you may need to restart the kernel to use updated packages.


In [51]:
import onnx
from onnxruntime.quantization import quantize_dynamic, QuantType

In [52]:
# Input ONNX model
model_fp32 = "cnn_model.onnx"

# Output quantized model
model_int8 = "cnn_model_quant.onnx"

quantize_dynamic(
    model_input=model_fp32,
    model_output=model_int8,
    weight_type=QuantType.QInt8  # quantize weights to INT8
)

In [53]:
import os

print("Original size:", os.path.getsize(model_fp32)/1024, "KB")
print("Quantized size:", os.path.getsize(model_int8)/1024, "KB")

Original size: 211.5859375 KB
Quantized size: 65.4609375 KB


In [62]:
import onnxruntime as ort
import numpy as np

# Load quantized model
session = ort.InferenceSession(model_int8)
input_name = session.get_inputs()[0].name

# Batch inference
pred = session.run(None, {input_name: X_test_proc.astype(np.float32)})
pred_labels = np.argmax(pred[0], axis=1)

# Accuracy
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, pred_labels)
print("Quantized ONNX model accuracy:", accuracy)

Quantized ONNX model accuracy: 0.9869
